In [1]:
!pip install -q tensorflow scikit-learn seaborn matplotlib
import numpy as np
import matplotlib.pyplot as plt
import os, cv2
from PIL import Image
import seaborn as sns
from sklearn.metrics import (top_k_accuracy_score, f1_score, roc_auc_score, auc, accuracy_score)
from sklearn.preprocessing import label_binarize
import tensorflow as tf
from tensorflow.keras.models import load_model
from google.colab import drive

In [2]:
drive.mount('/content/drive', force_remount=True)
MODEL_PATH_1 = '/content/drive/MyDrive/FineTuned01-EfficientNetB0_CNN_Model.h5'
MODEL_PATH_2 = '/content/drive/MyDrive/MobileNetV2_CNN_Model.keras'

Mounted at /content/drive


In [3]:
model_efficientnet = load_model(MODEL_PATH_1, safe_mode = False)
model_mobilenet = load_model(MODEL_PATH_2, safe_mode = False)
print("Models loaded!")

Models loaded!


In [4]:
DATA_FOLDER = '/content/drive/MyDrive/LandMark Images Pre-ProcessedUpdated'

**Data Loading**

In [5]:
# Load images
def load_images():
    splits = ['train', 'test', 'valid']
    X_train, y_train = [], []
    X_val, y_val = [], []
    X_test, y_test = [], []

    # Get landmark names
    classes = []
    for item in os.listdir(DATA_FOLDER):
        item_path = os.path.join(DATA_FOLDER, item)
        if os.path.isdir(item_path):
            subfolders = os.listdir(item_path)
            if all(split in subfolders for split in splits):
                classes.append(item)

    classes = sorted(classes)
    print(f"📁 Found {len(classes)} landmarks")

    # Create class to index mapping
    class_to_idx = {name: i for i, name in enumerate(classes)}

    # Load images from each split
    for landmark in classes:
        for split in splits:
            source_path = os.path.join(DATA_FOLDER, landmark, split)
            if not os.path.exists(source_path):
                continue

            images = [f for f in os.listdir(source_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]

            for img_name in images:
                try:
                    img_path = os.path.join(source_path, img_name)
                    img = np.array(Image.open(img_path).convert('RGB'))
                    img_resized = cv2.resize(img, (290, 290)) / 255.0

                    if split == 'train':
                        X_train.append(img_resized)
                        y_train.append(class_to_idx[landmark])
                    elif split == 'valid':
                        X_val.append(img_resized)
                        y_val.append(class_to_idx[landmark])
                    elif split == 'test':
                        X_test.append(img_resized)
                        y_test.append(class_to_idx[landmark])
                except:
                    pass

    # Convert to numpy arrays
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_val = np.array(X_val)
    y_val = np.array(y_val)
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    print(f"✅ Loaded: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")
    return X_train, X_val, X_test, y_train, y_val, y_test, classes

In [6]:
X_train, X_val, X_test, y_train, y_val, y_test, CLASS_NAMES = load_images()

📁 Found 24 landmarks
✅ Loaded: Train=501, Val=47, Test=24


In [7]:
X_eval = np.vstack([X_val, X_test])
y_eval = np.hstack([y_val, y_test])

print(f"\n📊 Evaluating on {len(X_eval)} images (Val + Test combined)")


📊 Evaluating on 71 images (Val + Test combined)


**Models** **Evaluation**

In [8]:
# Function used to evaluate both cnn models
def evaluate_model(model, X_eval, y_eval, CLASS_NAMES, model_name):
    print(f"\n{'='*60}")
    print(f"EVALUATING: {model_name}")
    print(f"{'='*60}")

    # Predictions
    tf.random.set_seed(42)
    y_pred_probs = model.predict(X_eval, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # 1. ACCURACY
    acc = accuracy_score(y_eval, y_pred)

    # 2. TOP-3 ACCURACY (Fixed: added labels argument)
    top3_acc = top_k_accuracy_score(y_eval, y_pred_probs, k=3, labels=range(len(CLASS_NAMES)))

    # 3. F1-SCORE
    f1 = f1_score(y_eval, y_pred, average='macro', zero_division=0)

    # 4. ROC-AUC
    y_eval_bin = label_binarize(y_eval, classes=range(len(CLASS_NAMES)))
    roc_auc = roc_auc_score(y_eval_bin, y_pred_probs, multi_class='ovr', average='macro')

    # Print Results
    print(f"1️⃣ ACCURACY:   {acc:.4f} ({acc*100:.2f}%)")
    print(f"2️⃣ TOP-3 ACC:  {top3_acc:.4f} ({top3_acc*100:.2f}%)")
    print(f"3️⃣ F1-SCORE:   {f1:.4f} ({f1*100:.2f}%)")
    print(f"4️⃣ ROC-AUC:    {roc_auc:.4f} ({roc_auc*100:.2f}%)")

    return {
        'name': model_name,
        'accuracy': acc,
        'top3_accuracy': top3_acc,
        'f1_score': f1,
        'roc_auc': roc_auc
    }

In [9]:
# Evaluating the EfficientNetB0 model
results_eff = evaluate_model(model_efficientnet, X_eval, y_eval, CLASS_NAMES, "EfficientNetB0")


EVALUATING: EfficientNetB0
1️⃣ ACCURACY:   0.7465 (74.65%)
2️⃣ TOP-3 ACC:  0.9296 (92.96%)
3️⃣ F1-SCORE:   0.7363 (73.63%)
4️⃣ ROC-AUC:    0.9705 (97.05%)


In [10]:
# Evaluating the MobileNetV2 model
results_mob = evaluate_model(model_mobilenet, X_test, y_test, CLASS_NAMES, "MobileNetV2")


EVALUATING: MobileNetV2
1️⃣ ACCURACY:   0.6667 (66.67%)
2️⃣ TOP-3 ACC:  0.8333 (83.33%)
3️⃣ F1-SCORE:   0.5903 (59.03%)
4️⃣ ROC-AUC:    0.9764 (97.64%)


In [11]:
print(f"\n{'='*60}")
print("                      FINAL COMPARISON")
print(f"{'='*60}")
print(f"{' Metric':<19} {'EfficientNetB0':<20} {'MobileNetV2':<18}")
print("-" * 60)
print(f"{'Accuracy':<22} {results_eff['accuracy']:<18.4f} {results_mob['accuracy']:<18.4f}")
print(f"{'Top-3 Accuracy':<22} {results_eff['top3_accuracy']:<18.4f} {results_mob['top3_accuracy']:<18.4f}")
print(f"{'F1-Score':<22} {results_eff['f1_score']:<18.4f} {results_mob['f1_score']:<18.4f}")
print(f"{'ROC-AUC':<22} {results_eff['roc_auc']:<18.4f} {results_mob['roc_auc']:<18.4f}")



                      FINAL COMPARISON
 Metric             EfficientNetB0       MobileNetV2       
------------------------------------------------------------
Accuracy               0.7465             0.6667            
Top-3 Accuracy         0.9296             0.8333            
F1-Score               0.7363             0.5903            
ROC-AUC                0.9705             0.9764            


In [12]:
best_model = results_eff if results_eff['accuracy'] > results_mob['accuracy'] else results_mob
print(f"\n   BEST MODEL: {best_model['name']}")
print(f"   Accuracy: {best_model['accuracy']:.4f}")
print(f"   Top-3 Accuracy: {best_model['top3_accuracy']:.4f}")
print(f"   F1-Score: {best_model['f1_score']:.4f}")
print(f"   ROC-AUC:  {best_model['roc_auc']:.4f}")


   BEST MODEL: EfficientNetB0
   Accuracy: 0.7465
   Top-3 Accuracy: 0.9296
   F1-Score: 0.7363
   ROC-AUC:  0.9705
